# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.mart_user_activation`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

c:\workspace\final_project\sns_service_analysis\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,signup_at,group_id,school_id,questionset_count,first_questionset_at,last_questionset_at,vote_count,first_vote_at,last_vote_at,selected_count,read_selected_count,first_selected_at,payment_count,first_payment_at,last_payment_at
0,843928,2023-04-23 15:42:58.053443+00:00,436,2043,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT
1,849256,2023-04-27 15:35:45.064387+00:00,3887,3515,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,1,2023-05-16 12:31:42+00:00,2023-05-16 12:31:42+00:00
2,882758,2023-05-05 00:29:47.354538+00:00,11043,5440,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT
3,914332,2023-05-06 13:01:18.898545+00:00,6182,5262,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT
4,917832,2023-05-06 13:47:05.670024+00:00,11747,2001,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT


## 데이터 확인

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   group_id              677080 non-null  Int64              
 3   school_id             677080 non-null  Int64              
 4   questionset_count     4972 non-null    Int64              
 5   first_questionset_at  4972 non-null    datetime64[us, UTC]
 6   last_questionset_at   4972 non-null    datetime64[us, UTC]
 7   vote_count            4849 non-null    Int64              
 8   first_vote_at         4849 non-null    datetime64[us, UTC]
 9   last_vote_at          4849 non-null    datetime64[us, UTC]
 10  selected_count        15426 non-null   Int64              
 11  read_selected_count   15426 non-null   Int64              
 12 

## 결측 처리
- 날짜 데이터를 제외한 결측은 0으로 대체

In [5]:
count_cols = [
    'questionset_count',
    'vote_count',
    'selected_count',
    'read_selected_count',
    'payment_count'
]

df[count_cols] = df[count_cols].fillna(0)

In [6]:
# 결측 대체 적용 체크

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   group_id              677080 non-null  Int64              
 3   school_id             677080 non-null  Int64              
 4   questionset_count     677080 non-null  Int64              
 5   first_questionset_at  4972 non-null    datetime64[us, UTC]
 6   last_questionset_at   4972 non-null    datetime64[us, UTC]
 7   vote_count            677080 non-null  Int64              
 8   first_vote_at         4849 non-null    datetime64[us, UTC]
 9   last_vote_at          4849 non-null    datetime64[us, UTC]
 10  selected_count        677080 non-null  Int64              
 11  read_selected_count   677080 non-null  Int64              
 12 

In [7]:
df[df['signup_at'] < '2023-04-28']['vote_count'].value_counts()

vote_count
0       8591
41         1
166        1
88         1
273        1
567        1
250        1
168        1
22         1
10         1
553        1
256        1
170        1
246        1
217        1
378        1
2786       1
244        1
338        1
141        1
212        1
131        1
Name: count, dtype: Int64

In [8]:
pd.crosstab(
    df['questionset_count'] > 0,
    df['vote_count'] > 0,
    margins=True
)

vote_count,False,True,All
questionset_count,,,
False,672108,0,672108
True,123,4849,4972
All,672231,4849,677080


# 질문 및 투표참여 비율 파악

- 투표기록에 기록이 없고, 질문세트에도 기록이 없는 유저가 672,108명
- 질문 세트 없이 투표한 유저는 없음.
- 질문 세트는 열었지만 투표 기록에는 없는 유저가 123명
- 질문 세트를 열고 투표기록까지 남긴 유저가 4,849명

해당 내용을 해석해보자면 총 677,080명 중 투표가 기록된 유저 4,849명 약 0.7% 정도로 본다면 실제 투표 참여율은 가히 충격적인 수치..
또한, 투표한 모든 유저가 반드시 questionset 경험 기록을 가지고 있다고 보이며, 질문세트를 열고 투표한 기록이 투표기록 테이블에 적재가 되는 순서로 진행되는 것으로 보이며,
질문 세트를 열었지만 투표는 하지 않은 유저가 123명으로 투표 과정 중 이탈이라고 볼 수 있다.

반대로 해석해보자면 질문을 참여하기까지의 프로세스가 원활하지 않거나, 유저 개인의 성향으로 질문 투표를 하지 않는 등의 사유로 콘텐츠 이용이 원활하지않다고 볼 수 있다.
따라서, 해당 해석 결과를 토대로 콘텐츠 경험 이후 병목 보다, 콘텐츠 이용이 진행되지 않은 원인을 분석하는 것이 무엇보다 중요하다고 판단된다.


In [9]:
print("<포인트 구매 유저의 특성 확인>")

print(f"포인트 구매 경험이 있는 유저 : {df[(df['payment_count'] > 0)]['user_id'].nunique():,}")
print(f"질문 투표를 해보지 않고 포인트 구매한 유저 : {df[(df['payment_count'] > 0) & (df['vote_count'] == 0) & (df['questionset_count'] == 0)]['user_id'].nunique():,}")

print(f"투표 선택된 경험이 있는 유저 : {df[(df['selected_count'] > 0)]['user_id'].nunique():,}")
print(f"투표 선택된 경험이 있으면서 포인트를 구매한 유저 : {df[(df['selected_count'] > 0) & (df['payment_count'] > 0)]['user_id'].nunique():,}")

<포인트 구매 유저의 특성 확인>
포인트 구매 경험이 있는 유저 : 59,192
질문 투표를 해보지 않고 포인트 구매한 유저 : 58,782
투표 선택된 경험이 있는 유저 : 15,426
투표 선택된 경험이 있으면서 포인트를 구매한 유저 : 1,213


# 포인트를 구매한 유저 특성 파악
- 질문 투표를 경험한 유저가 약 4,800여명 정도이나, 포인트를 구매한 경험이 있는 유저가 59,192명

핵심 콘텐츠를 이용하지 않으면서 포인트를 결제한 유저가 다수 존재하며, 포인트 사용처가 대부분 투표를 보낸사람의 초성을 확인하거나, 채팅방을 여는 것에 사용하는 것으로 확인할 수 있었기 때문에 선행 조건인 투표를 받은 경험이 있는 유저가 포인트를 구매하는지 확인하기 위해 '투표를 받아본 경험이 있으면서, 포인트를 구매한 경험이 있는 유저'를 조건으로 확인한 결과 1,213명으로 예상과는 다르게 다른 이유로 결제한 유저가 더 많다는 것을 간접적으로 확인이 가능했다.

따라서, 투표에 선택된 경험이 포인트 구매에 미치는 영향이 크지 않다고 볼 수 있다.

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   group_id              677080 non-null  Int64              
 3   school_id             677080 non-null  Int64              
 4   questionset_count     677080 non-null  Int64              
 5   first_questionset_at  4972 non-null    datetime64[us, UTC]
 6   last_questionset_at   4972 non-null    datetime64[us, UTC]
 7   vote_count            677080 non-null  Int64              
 8   first_vote_at         4849 non-null    datetime64[us, UTC]
 9   last_vote_at          4849 non-null    datetime64[us, UTC]
 10  selected_count        677080 non-null  Int64              
 11  read_selected_count   677080 non-null  Int64              
 12 

In [12]:
df['school_id'].value_counts()

school_id
369     578
1719    551
4516    510
5372    507
5520    500
       ... 
3247      1
4992      1
4209      1
3668      1
1324      1
Name: count, Length: 5551, dtype: Int64

# 각 유저 가입 시점에 서비스 제한여부 확인

In [ ]:
# 학교 아이디별 가입 순서 컬럼 생성

df['school_signup_order'] = (
    df.groupby('school_id')['signup_at']
      .rank(method='first')
)

In [ ]:
# 학교별 가입 순서가 40번째인 학교 아이디와 40명이 충족된 시점 집계

school_unlock = (
    df[df['school_signup_order'] == 40]
    [['school_id', 'signup_at']]
    .rename(columns={'signup_at': 'school_unlock_at'})
)

school_unlock.head()

,school_id,school_unlock_at
55,3080,2023-05-15 11:37:23.834978+00:00
370,4121,2023-05-17 07:47:16.155207+00:00
384,982,2023-05-22 01:25:46.967848+00:00
621,1707,2023-05-07 15:43:10.176955+00:00
1040,3315,2023-05-15 09:36:10.202319+00:00


In [ ]:
# 그룹(학급)별 4번째 가입한 시점을 집계

df = df.sort_values('signup_at').copy()

df['group_signup_order'] = (
    df.groupby('group_id')
      .cumcount() + 1
)

group_unlock = (
    df[df['group_signup_order'] == 4]
    [['group_id', 'signup_at']]
    .rename(columns={'signup_at': 'group_unlock_at'})
)

group_unlock.head()

,group_id,group_unlock_at
660911,12,2023-03-29 13:20:46.429584+00:00
292324,1,2023-03-29 13:20:46.476550+00:00
525020,40,2023-03-31 16:36:32.784004+00:00
401774,42,2023-04-01 03:21:05.685892+00:00
161046,41,2023-04-02 01:22:17.883572+00:00


In [18]:
df = df.merge(
    school_unlock,
    on='school_id',
    how='left'
)

df = df.merge(
    group_unlock,
    on='group_id',
    how='left'
)

In [20]:
df[
    [
        'user_id',
        'signup_at',
        'school_id',
        'group_id',
        'school_unlock_at',
        'group_unlock_at'
    ]
].head()

,user_id,signup_at,school_id,group_id,school_unlock_at,group_unlock_at
0,831962,2023-03-29 05:18:56.162368+00:00,1,12,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00
1,832151,2023-03-29 12:56:34.989468+00:00,1,1,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.476550+00:00
2,832340,2023-03-29 12:56:35.020790+00:00,1,1,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.476550+00:00
3,832520,2023-03-29 12:56:35.049311+00:00,1,12,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00
4,832614,2023-03-29 12:56:35.064406+00:00,1,12,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00


In [21]:
df[['school_unlock_at', 'group_unlock_at']].isna().sum()

school_unlock_at    21580
group_unlock_at     40487
dtype: int64

In [24]:
# 실제 이용이 가능했던 시점

df['service_available_at'] = (
    df[['school_unlock_at', 'group_unlock_at']]
    .max(axis=1, skipna=False)
)

In [ ]:
# 결과 체크!

df[
    [
        'signup_at',
        'school_unlock_at',
        'group_unlock_at',
        'service_available_at'
    ]
].head()

,signup_at,school_unlock_at,group_unlock_at,service_available_at
0,2023-03-29 05:18:56.162368+00:00,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00,2023-04-29 23:56:29.729867+00:00
1,2023-03-29 12:56:34.989468+00:00,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.476550+00:00,2023-04-29 23:56:29.729867+00:00
2,2023-03-29 12:56:35.020790+00:00,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.476550+00:00,2023-04-29 23:56:29.729867+00:00
3,2023-03-29 12:56:35.049311+00:00,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00,2023-04-29 23:56:29.729867+00:00
4,2023-03-29 12:56:35.064406+00:00,2023-04-29 23:56:29.729867+00:00,2023-03-29 13:20:46.429584+00:00,2023-04-29 23:56:29.729867+00:00


In [26]:
# 유저별 그룹으로 분류
# 가임당시 이미 사용이 가능했던 유저, 가입 당시 이용이 불가했지만 충족으로 변경된 유저, 데이터 기간내에 계속 불가인 유저

conditions = [
    df['service_available_at'].isna(),
    df['signup_at'] >= df['service_available_at'],
    df['signup_at'] < df['service_available_at']
]

choices = [
    'unavailable',
    'available_at_signup',
    'available_later'
]

df['service_access_status'] = np.select(
    conditions,
    choices,
    default='unknown'
)

In [33]:
status_count = df['service_access_status'].value_counts()
status_pct = df['service_access_status'].value_counts(normalize=True).mul(100).round(2)

print(f"가입 즉시 이용 가능: {status_count['available_at_signup']:,}명 ({status_pct['available_at_signup']:.2f}%)")
print(f"가입 후 이용 가능: {status_count['available_later']:,}명 ({status_pct['available_later']:.2f}%)")
print(f"관측기간 내 이용 불가: {status_count['unavailable']:,}명 ({status_pct['unavailable']:.2f}%)")

가입 즉시 이용 가능: 410,441명 (60.62%)
가입 후 이용 가능: 215,915명 (31.89%)
관측기간 내 이용 불가: 50,724명 (7.49%)


## 서비스 제한 관련 해체 시점과 현황을 확인한 결과.

- 전체 가입자의 약 60.6%는 가입 시점부터 서비스 이용 조건을 이미 충족.
- 반면 약 31.9%는 가입 당시에는 인원 조건을 충족하지 못해 서비스를 바로 이용할 수 없었으며, 이후 학교·학급 인원이 증가하면서 이용 가능한 상태가 된 것으로 확인 됨.
- 관측기간 동안에도 조건을 충족하지 못한 유저는 약 **7.5%**

따라서, 질문 경험자가 약 5천 명에 불과한 현상을 단순히 학교·학급 인원 미달만으로 설명하기는 어렵다고 볼 수 있으며,
특히 410,441명(60.62%)은 가입 시점부터 인원 조건을 충족한 상태였음에도 전체 질문 경험자는 약 5천 명 수준으로 확인된다.

향후에는 다음과 같은 추가적인 Activation 병목이 존재하는지 확인할 필요가 있다.

**가입 -> 서비스 이용 조건 충족 -> 홈 진입 -> 질문 시작 -> 질문 세트 경험 -> 투표 -> 투표완료**

단, 가입당시 이용이 불가능 했으나 이후 이용 가능 그룹으로 전환된 215,915명은 별도로 어느정도 기간동안 기다려야했는지, 또는 실제로 이용 이력이 있는지 당시 이용불가로 이탈하지 않았는지 추가 분석이 필요하다.